# 03 — Git pour développeurs SVN

Ce notebook couvre les différences fondamentales entre SVN et Git.  
Objectif : comprendre Git en profondeur, pas juste mémoriser des commandes.

**Prérequis :** avoir complété `02-conda.ipynb`

## 1. Architecture fondamentale : centralisé vs distribué

### SVN — Architecture centralisée

```
        [SERVEUR SVN]
        ┌───────────┐
        │ Dépôt     │  ← L'unique source de vérité
        │ complet   │
        │ r1, r2... │
        └─────┬─────┘
              │ réseau (HTTP, SVN://)
    ┌─────────┼─────────┐
    │         │         │
┌───┴───┐ ┌───┴───┐ ┌───┴───┐
│Alice  │ │Bob    │ │Carol  │
│working│ │working│ │working│
│copy   │ │copy   │ │copy   │
│(partiel)│ │(partiel)│ │(partiel)│
└───────┘ └───────┘ └───────┘
```

Chaque développeur a une **working copy partielle** — juste les fichiers du répertoire courant.  
L'historique complet vit uniquement sur le serveur.  
Si le serveur tombe : tu peux lire tes fichiers locaux, mais tu ne peux pas consulter l'historique, créer une branche, ni commiter.

---

### Git — Architecture distribuée

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ Alice        │     │ GitHub       │     │ Bob          │
│ ┌──────────┐ │     │ ┌──────────┐ │     │ ┌──────────┐ │
│ │ Repo     │ │     │ │ Repo     │ │     │ │ Repo     │ │
│ │ COMPLET  │◄├─────┤►│ COMPLET  │◄├─────┤►│ COMPLET  │ │
│ │ + history│ │push/│ │ + history│ │push/│ │ + history│ │
│ └──────────┘ │pull │ └──────────┘ │pull │ └──────────┘ │
└──────────────┘     └──────────────┘     └──────────────┘
```

Chaque développeur possède un **clone complet** — tout l'historique depuis le début du projet.  
GitHub n'est qu'un repo parmi d'autres, désigné comme "remote" par convention.  
Tu peux travailler offline : consulter l'historique, créer des branches, commiter — tout sans réseau.

| Opération | SVN | Git |
|-----------|-----|-----|
| Voir l'historique | `svn log` → requête réseau | `git log` → lecture locale |
| Créer une branche | opération réseau | opération locale |
| Commiter | envoi immédiat au serveur | stockage local |

## 2. Modèle de données : deltas vs snapshots

### SVN — Stockage par delta (différences successives)

SVN stocke **ce qui a changé** entre chaque révision. Pour reconstruire l'état à r5, SVN rejoue la chaîne depuis le début — le delta *est* le modèle de données.

```
r1: [contenu complet de hello.py]
r2: [delta: ligne 3 modifiée]
r3: [delta: lignes 7-9 ajoutées]
r4: [delta: ligne 3 supprimée]
r5: [delta: ligne 1 modifiée]

Pour lire hello.py à r5 : r1 + delta(r2) + delta(r3) + delta(r4) + delta(r5)
```

---

### Git — Stockage par snapshot complet

Git stocke le **fichier complet** à chaque fois qu'il change. Si un fichier n'a pas changé entre deux commits, Git stocke un pointeur vers le snapshot existant — pas de duplication.

```
Commit A :  [hello.py complet v1] [utils.py complet v1]
                │                        │
Commit B :  [pointeur → v1]       [utils.py complet v2]  ← utils.py a changé
                │                                 │
Commit C :  [hello.py complet v2] [pointeur → v2]         ← hello.py a changé
```

**Analogie C++ :** SVN, c'est reconstruire une `struct` en appliquant une série de patches.  
Git, c'est lire la `struct` directement en mémoire — l'accès est immédiat, peu importe la profondeur de l'historique.

---

### Est-ce que Git prend plus d'espace disque ?

En théorie, un snapshot complet prend plus de place qu'un delta. Mais Git compense avec les **packfiles** : une compression interne automatique qui rapproche les snapshots de la taille de deltas. Cette compression est plus agressive que celle de SVN car Git peut comparer des fichiers *différents* entre eux, pas seulement les révisions du même fichier.

En pratique, les deux systèmes sont dans le même ordre de grandeur. L'exception : les **fichiers binaires** (images, modèles ML) — un binaire qui change souvent génère un snapshot complet inutilisable par la compression delta. C'est pour ça qu'existe Git LFS, mais c'est un sujet séparé.

---

### Complexité d'accès : notation Big O

Le tableau ci-dessous utilise la notation **Big O**, standard en informatique pour exprimer la quantité de travail selon la taille des données.

- **O(n)** : le travail croît proportionnellement à `n`. Ici, `n` = le nombre de révisions. Plus le repo est vieux, plus c'est lent.
- **O(1)** : le travail est constant, peu importe la taille. En C, c'est l'équivalent d'accéder à `tableau[i]` — calcul d'adresse direct, toujours le même coût.

| | SVN | Git |
|--|-----|-----|
| Modèle de données | Delta (rejoué à chaque lecture) | Snapshot complet |
| Accès à un état ancien | O(n) — rejoue tous les deltas | O(1) — lit le snapshot directement |
| Compression interne | Delta par fichier | Packfiles (delta cross-fichiers) |
| Espace disque | Comparable en pratique | Comparable en pratique |

# 3. Les branches : copie physique vs pointeur léger

---

## SVN — Une branche est une copie de répertoire

En SVN, une branche est littéralement une copie du répertoire dans le dépôt :

```
Dépôt SVN :
  /trunk/           ← développement principal
  /branches/
    /feature-auth/  ← copie de /trunk au moment du branch
    /bugfix-42/     ← copie de /trunk au moment du branch
  /tags/
    /v1.0/          ← copie figée (convention, pas protégée)
```

```bash
# SVN : créer une branche = copie réseau côté serveur
svn copy https://svn.example.com/trunk \
         https://svn.example.com/branches/feature-auth \
         -m "Création branche feature-auth"
# → opération réseau, crée une révision, copie les fichiers côté serveur
```

SVN utilise une **lazy copy** (copie sur écriture côté serveur) : à la création de la branche, aucun fichier n'est copié physiquement. SVN crée des **références vers les nodes existants** du dépôt — exactement comme `std::string` avec COW en C++. La copie réelle n'arrive que lors du premier commit sur un fichier donné.

**Au moment de la copie :** SVN ne copie *rien* physiquement. Il crée juste un **pointeur** (une référence) vers les fichiers originaux dans le dépôt. C'est instantané et ne prend presque pas de place.

```
trunk/                →  [fichier A, révision 42]
                          [fichier B, révision 42]

branches/ma-branche/  →  pointe vers les mêmes blocs
```

**Au moment d'une modification :** C'est seulement quand tu **modifies** un fichier dans la branche que SVN crée une vraie copie de ce fichier, uniquement pour celui-là.

```
Dépôt SVN après svn copy :

/trunk/                       → node_id: 42 (contenu réel)
/branches/feature-auth/       → node_id: 42 (même pointeur — rien n'est copié)

Premier commit dans feature-auth :
/branches/feature-auth/hello.py  → node_id: 99 (nouveau node, contenu réel)
/branches/feature-auth/utils.py  → node_id: 42 (toujours le même pointeur)
```

La branche ne coûte presque rien à la création. Le coût en espace disque arrive **au fil des commits**, fichier par fichier.

**Pourquoi "lazy" (paresseux) ?** Le terme vient du fait que SVN **reporte le travail** au dernier moment possible — il ne fait la vraie copie que si c'est absolument nécessaire (à l'écriture). C'est un principe de programmation classique : *"ne fais pas maintenant ce que tu peux éviter de faire"*.

**Les bénéfices concrets :**

| Avantage | Explication |
|---|---|
| 🚀 **Rapidité** | Créer une branche est quasi-instantané, peu importe la taille du projet |
| 💾 **Espace disque** | Le dépôt ne grossit que proportionnellement aux *vraies* modifications |
| 🏷️ **Tags bon marché** | Un tag (snapshot) ne coûte presque rien à créer |

---

## Le problème de merge avant SVN 1.5 — et ce qu'il révèle

Pour fusionner deux branches, il faut savoir exactement quels deltas appliquer — ni trop, ni trop peu. Avant SVN 1.5, SVN ne stockait **nulle part** l'information "j'ai déjà mergé jusqu'à la révision r42". Cette métadonnée n'existait tout simplement pas dans le modèle de données.

```
trunk:   r1 ── r2 ── r3 ── r4 ── r5 ── r6
                      │                  ↑
                      │            tu veux merger ici
                   branch:
                      └── r3 ── r4 ── r5

Question : quels deltas appliquer sur trunk ?
→ r4 et r5 de la branche uniquement (r3 est le point de divergence)
→ SVN < 1.5 ne savait pas ça automatiquement — c'était à TOI de t'en souvenir
```

En pratique, tu devais noter manuellement les numéros de révision déjà mergés et spécifier la plage à la main :

```bash
# SVN < 1.5 : merger manuellement en précisant la plage exacte
svn merge -r 3:5 https://svn.example.com/branches/feature-auth
# Si tu oubliais ou te trompais → conflits ou double-application de deltas
```

Ce n'est pas un simple bug ou un oubli de développeur — c'est la **conséquence directe du modèle de données de SVN**. Puisqu'une branche est juste une copie de répertoire dans un dépôt à révisions globales linéaires, SVN n'a aucun moyen *structurel* de savoir d'où vient quoi. Il faut compenser par une métadonnée externe.

SVN 1.5 a introduit les **mergeinfo** — une propriété `svn:mergeinfo` stockée dans les fichiers qui enregistre automatiquement les révisions déjà mergées. Ça a résolu le problème en pratique, mais c'est une métadonnée de compensation ajoutée après coup, fragile à manipuler.

> **La leçon sous-jacente :** un mauvais modèle de données force à ajouter des métadonnées externes pour compenser ses lacunes. Un bon modèle de données rend ces métadonnées inutiles.

---

## Git — Une branche est un pointeur de 41 octets

En Git, une branche n'est qu'un **fichier texte contenant le hash SHA-1** du commit pointé :

```
.git/refs/heads/
  main          ← fichier contenant "a3f8c2d1..."
  feature-auth  ← fichier contenant "7b2e9f4c..."
  bugfix-42     ← fichier contenant "c1d4a8e2..."
```

```bash
# Git : créer une branche = écrire 41 octets sur disque
git branch feature-auth
# ou, créer et basculer dessus immédiatement :
git checkout -b feature-auth
# → opération locale, instantanée, zéro réseau
```

```
Historique des commits :

a3f8c2d ← main (pointe ici)
    │
7b2e9f4 ← feature-auth (pointe ici)
    │
c1d4a8e
    │
b9f3d1c
```

**Switching de branche :** Git met à jour les fichiers du working directory pour correspondre au snapshot du commit pointé. Il ne "copie" rien — il restaure un snapshot.

**Conséquence sur les merges :** Git n'a jamais eu le problème de SVN. Chaque commit connaît son parent via le hash SHA-1 — l'ancêtre commun de deux branches se calcule nativement en remontant le graphe de commits. C'est **structurel**, pas une métadonnée ajoutée. Ce n'est pas que Git a "mieux codé" la même chose : c'est que Git a **pensé le merge dès la conception du modèle de données**.

---

## Comparaison finale

| | SVN | Git |
|---|---|---|
| **Une branche, c'est quoi ?** | Une copie de répertoire | Un pointeur de 41 octets |
| **Coût de création** | Quasi-nul (lazy copy) | Quasi-nul (41 octets) |
| **Opération** | Réseau + révision serveur | Locale, zéro réseau |
| **Merge tracking** | Métadonnée externe (`svn:mergeinfo`) ajoutée en SVN 1.5 | Structurel via le graphe de commits SHA-1 |
| **Ancêtre commun** | Calculé à partir d'une métadonnée fragile | Calculé nativement en remontant le DAG |

La création d'une branche coûte presque rien dans les deux cas — ce n'est pas là que ça diverge vraiment. La vraie ligne de fracture, c'est le merge tracking :

- **SVN** : modèle linéaire (révisions globales) → merge tracking **impossible nativement** → compensation par métadonnée externe
- **Git** : modèle graphe (commits avec parents) → merge tracking **gratuit et structurel**

## 4. Workflow de commit : SVN en 1 étape vs Git en 2 étapes

### SVN — Commit direct au serveur

```
Working copy  ──── svn commit ────►  Serveur SVN
(fichiers modifiés)                  (nouvelle révision r42, visible par tous)
```

```bash
# SVN : modifier puis commiter = publier immédiatement
nano hello.py
svn commit -m "Fix bug in hello.py"
# → requête réseau, génère r42 sur le serveur, visible par tous immédiatement
```

Sans accès réseau → impossible de commiter.  
Commit trop tôt → tout le monde voit ton travail inachevé.

---

### Git — Commit local, puis push séparé

```
Working       ──── git add ────►  Staging  ──── git commit ────►  Repo local  ──── git push ────►  Remote
directory                          Area                            (.git/)                          (GitHub)
(fichiers modifiés)               (index)                         (nouveau commit)
```

```bash
# Git : modifier, stager, commiter localement, puis pousser quand prêt
nano hello.py
nano utils.py

git add hello.py              # staging area : seulement hello.py
git commit -m "Fix bug"       # commit LOCAL — personne ne voit encore
# ... continuer à travailler, faire d'autres commits ...
git push origin main          # maintenant GitHub est mis à jour
```

La séparation commit/push est puissante : tu peux faire 10 commits locaux pour affiner  
ton travail, puis les envoyer en une fois quand ils sont prêts.

## 5. La Staging Area — concept absent en SVN

C'est le concept le plus déroutant pour quelqu'un venant de SVN, car il n'a pas d'équivalent.

### Qu'est-ce que c'est ?

La Staging Area (aussi appelée **Index**) est une zone intermédiaire entre le working directory  
et le repo Git. C'est une **sélection explicite de ce qui va faire partie du prochain commit**.

```
┌─────────────────┐    git add    ┌─────────────────┐   git commit   ┌─────────────────┐
│ Working         │ ─────────────►│ Staging Area    │ ──────────────►│ Repository      │
│ Directory       │               │ (Index)         │                │ (.git/)         │
│                 │               │                 │                │                 │
│ hello.py   (M)  │──── add ─────►│ hello.py   (M)  │                │                 │
│ utils.py   (M)  │  (pas stagé)  │                 │                │                 │
│ config.py  (M)  │──── add ─────►│ config.py  (M)  │                │                 │
└─────────────────┘               └─────────────────┘                └─────────────────┘
                                  ↑
                          Seulement ces deux fichiers
                          iront dans le commit
```

### Pourquoi c'est utile ?

Tu as modifié 5 fichiers en travaillant sur deux choses différentes (un bug fix et une nouvelle feature).  
En SVN → tu commites tout ensemble.  
En Git → tu fais deux commits séparés et propres :

```bash
# Tu as modifié : auth.py, utils.py (bug fix) + ui.py, templates/ (nouvelle feature)

# Commit 1 — seulement le bug fix
git add auth.py utils.py
git commit -m "Fix authentication bug #42"

# Commit 2 — seulement la feature
git add ui.py templates/
git commit -m "Add user profile page"
```

### États des fichiers en Git

```
Untracked  ──── git add ────►  Staged  ──── git commit ────►  Committed
(nouveau fichier,              (dans l'index,                 (dans le repo,
 Git ne le connaît pas)         prêt pour le commit)           snapshot créé)

Modified   ──── git add ────►  Staged
(fichier connu, contenu changé)
```

```bash
git status           # voir l'état de chaque fichier
git diff             # différences working directory vs staging area
git diff --staged    # différences staging area vs dernier commit
```

## 6. Commit local vs serveur — Identifiants et visibilité

### SVN — Le numéro de révision est global et séquentiel

En SVN, chaque révision a un numéro entier **global et séquentiel** assigné par le serveur : r1, r2, r3...  
Il est impossible d'avoir un commit "local" — commiter = publier immédiatement.

```
r1  ── r2  ── r3  ── r4  ── r5
                              ↑
                         dernier commit
                         (visible par tous dès que créé)
```

### Git — Le hash SHA-1 est universel et calculé localement

En Git, chaque commit est identifié par un **hash SHA-1** calculé à partir de :
- le contenu des fichiers snapshottés
- le nom et email de l'auteur
- la date et heure du commit
- le hash du commit parent

Ce hash est **identique sur toutes les machines** qui ont ce commit — sans coordination avec un serveur.

```bash
git log --oneline
a3f8c2d Fix bug in auth module        ← commit local, pas encore pushé
7b2e9f4 Add user profile feature      ← commit local, pas encore pushé
c1d4a8e Initial project structure     ← commit déjà sur GitHub
```

**Conséquence :** Deux développeurs peuvent travailler offline une semaine, créer des dizaines  
de commits chacun, puis synchroniser — Git peut fusionner leurs historiques car les hashes  
garantissent l'unicité globale de chaque commit sans serveur central.

| Aspect | SVN | Git |
|--------|-----|-----|
| Identifiant de commit | `r42` (entier séquentiel serveur) | `a3f8c2d` (SHA-1 universel) |
| Offline commit | Impossible | Natif |
| Offline log | Impossible | Natif |
| Visibilité après commit | Immédiate pour tous | Locale uniquement jusqu'au push |

## 7. Tableau récapitulatif

| Concept | SVN | Git |
|---------|-----|-----|
| **Architecture** | Centralisée — un seul serveur | Distribuée — chaque clone est complet |
| **Historique local** | Non — sur le serveur uniquement | Oui — complet sur chaque machine |
| **Modèle de stockage** | Deltas (différences successives) | Snapshots (état complet du projet) |
| **Branches** | Copie de répertoire côté serveur | Pointeur de 41 octets, local |
| **Création de branche** | Opération réseau, côté serveur | Instantanée, locale |
| **Identifiant de commit** | Entier séquentiel global (`r42`) | Hash SHA-1 universel (`a3f8c2d`) |
| **Commit** | = publication immédiate au serveur | = sauvegarde locale uniquement |
| **Publication** | Implicite dans `svn commit` | Explicite via `git push` |
| **Staging Area** | Inexistante | Obligatoire (index) |
| **Travail offline** | Lecture seule | Complet (commit, branch, log...) |
| **Merge** | Difficile (historique mal tracké avant SVN 1.5) | Fiable (ancêtre commun connu) |
| **Checkout d'un commit ancien** | Lent — rejoue tous les deltas | Rapide — lecture directe du snapshot |

## 8. Métaphore finale

**SVN, c'est Google Docs :**  
Le document vit sur le serveur. Tu ouvres une connexion, tu modifies, tu sauvegardes —  
et tout le monde voit la sauvegarde immédiatement. Sans connexion : lecture seule.

**Git, c'est un carnet de notes physique avec photocopieuse :**  
Chaque développeur a son carnet complet. Tu écris dedans (commit local) sans connexion.  
Quand tu veux partager, tu photocopies les nouvelles pages et tu les envoies (push).  
Les autres font pareil et vous fusionnez vos carnets (merge/pull).  
Le carnet partagé (GitHub) n'est qu'un carnet désigné comme référence par convention —  
il n'a aucun statut spécial techniquement.